# Instrument characteristics

The CDI forms were built separately for each language, so language and instrument are confounded. This table puts them side by side. Input predictors are added once the corpus side has them.

Wordbank's own ages of acquisition are cut off at the edges of each form. If ours agree with them, the cut-off cannot explain differences between languages.

In [ ]:
import os
import sys

# notebooks live one level down, src is imported from the project root
if "src" not in os.listdir():
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import pandas as pd
from src.aoa import administrations, exclusions, items
from src.common import schema
from src.common.config import LANGUAGES, TABLES, raw_dir

In [ ]:
wordbank = pd.read_parquet(raw_dir("wordbank") / "aoa.parquet")
wordbank = wordbank[wordbank["measure"] == "produces"]

rows = []
for language, name in LANGUAGES.items():
    admins = administrations.load(language)
    words = items.load(language)
    low, high = exclusions.age_range(language)

    row = {
        "language": language,
        "children": len(admins),
        "ages": admins["age"].nunique(),
        "low": low,
        "high": high,
        "items": len(words),
    }

    # Finnish was excluded and has no handover file
    if schema.path(language).exists():
        ours = schema.read(language)
        row["lemmas"] = len(ours)
        row["below"] = (ours["aoa"] < low).sum()
        row["above"] = (ours["aoa"] > high).sum()
        row["extrapolated"] = (row["below"] + row["above"]) / len(ours)

        for norm in schema.NORMS:
            row[f"sd_{norm}"] = ours[norm].std()

        # Wordbank fits every item on every form on its own, the earliest item stands for the lemma
        mine = words[words["uni_lemma"].isin(ours["uni_lemma"])]
        theirs = wordbank.loc[wordbank["language"] == name, ["form", "item_id", "aoa"]]
        theirs = mine.merge(theirs, on=["form", "item_id"]).groupby("uni_lemma")["aoa"].min()

        both = ours.set_index("uni_lemma")["aoa"].to_frame("ours").join(theirs.rename("wordbank"))
        row["wordbank_lemmas"] = both["wordbank"].notna().sum()
        row["wordbank_r"] = both["ours"].corr(both["wordbank"])

    rows.append(row)

table = pd.DataFrame(rows)

TABLES.mkdir(parents=True, exist_ok=True)
table.to_csv(TABLES / "instruments.csv", index=False)
table.round(2)

## What the table shows

Finnish has 70 children at only two ages, too few for a curve, so it has no handover file.

Our ages of acquisition correlate with Wordbank's at 0.97 to 0.99, and at most 6 lemmas per language fall below the range. The cut-off at the lower edge changes little.

Spanish is the exception at the top. 72 of its 289 lemmas lie above 30 months, and Wordbank has values for only 191 of them.

The norms spread about equally in all languages.